# Critical Force — Measuring Your Aerobic Climbing Capacity

> *How much force can your fingers sustain indefinitely?*
> *And how big is your anaerobic battery?*

This notebook explains the **Critical Force (CF)** model — the gold standard
for measuring aerobic endurance capacity of the finger flexors in climbing.

**What you will learn:**

1. What Critical Force and W′ mean physiologically
2. How to perform a 3-point test on a hangboard
3. How to compute CF and W′ from test data
4. How to interpret your results and predict time-to-failure
5. What training methods improve each component

## Background — The Critical Power Concept

In 1965, **Monod & Scherrer** described the *critical power* model for
sustained muscular work. The key insight: for any muscle group there exists
a **threshold intensity** below which exercise can be sustained
"indefinitely" (practically: for a very long time without failure). Above
this threshold, a finite **anaerobic work capacity** (W′) is progressively
depleted, and exhaustion occurs when W′ reaches zero.

**Giles et al. (2019)** adapted this framework to intermittent isometric
finger contractions on a hangboard — creating the **Critical Force (CF)**
test for climbing. The mathematics are identical to critical power, but
applied to grip force rather than cycling wattage:

$$\text{Work} = CF \times t + W'$$

where:
- **CF** (% MVC) = the highest force sustainable without depleting W′ — your *aerobic threshold*
- **W′** (% MVC · s) = the total anaerobic work capacity above CF — your *battery*
- **t** = cumulative hang time to exhaustion (seconds)

This is a simple linear model: plot cumulative work (intensity × time) against
time, and the **slope is CF**, the **intercept is W′**.

*References: Monod & Scherrer 1965; Poole et al. 2016; Giles et al. 2019.*

## The 3-Point Test Protocol

To determine CF and W′ you need **at least three exhaustive trials** at
different intensities. The standard protocol (Giles et al. 2019):

### Equipment
- Hangboard with a consistent edge (20 mm recommended)
- Force measurement device (Tindeq Progressor, force plate, or known added/removed weight)
- Timer / metronome for work:rest cadence (typically 7 s on : 3 s off)

### Steps

1. **Determine your MVC-7** — a 7-second maximal voluntary contraction on
   the test edge. Use the best of 2–3 attempts with full rest between.

2. **Choose 3 intensities** well above and well below expected CF:
   - **High**: ~80 % MVC (expect failure in ~1–2 minutes)
   - **Medium**: ~60 % MVC (expect failure in ~2–4 minutes)
   - **Low**: ~45 % MVC (expect failure in ~4–8 minutes)

3. **Hang until failure** at each intensity using a 7 s on / 3 s off
   cadence. Record the **cumulative hang time** (sum of all 7 s work
   bouts, excluding rest).

4. **Rest 20+ minutes** between trials (or test on separate days).

5. Fit the linear work–time model to extract CF and W′.

> ⚠️ The lowest intensity must be close enough to CF that you actually fail
> within ~8 minutes. If you can hang "forever", the intensity is below CF
> and the trial is uninformative.

## Your Data

Enter your test results below. Adjust the values to match your own
MVC-7, bodyweight, and 3-point test outcomes.

In [ ]:
%matplotlib inline

MVC7_KG = 105          # Your MVC-7 in kg (on the test edge)
BODYWEIGHT_KG = 72     # Your bodyweight in kg

# 3-point test results
# Each trial: hang at a fixed %MVC until failure, record cumulative hang time
INTENSITIES = [80, 60, 45]    # %MVC for each trial
DURATIONS   = [77, 136, 323]  # cumulative hang time in seconds

MVC7_PCT_BW = (MVC7_KG / BODYWEIGHT_KG) * 100
print(f"MVC-7: {MVC7_KG} kg  ({MVC7_PCT_BW:.0f} % BW)")
print(f"Trials: {list(zip(INTENSITIES, DURATIONS))}")

## Step 1 — Compute Critical Force and W′

The `critical_force()` function fits the linear work–time model:

$$\text{Work}_i = \text{intensity}_i \times t_i = CF \times t_i + W'$$

via ordinary least squares regression. It returns:
- **cf_percent_mvc**: Critical Force as % of your MVC
- **w_prime_pct_s**: W′ in units of %MVC · seconds
- **r_squared**: goodness of fit (should be ≥ 0.98 for valid data)

In [ ]:
from climbing_science.endurance import critical_force

result = critical_force(INTENSITIES, DURATIONS)

cf  = result["cf_percent_mvc"]
wp  = result["w_prime_pct_s"]
r2  = result["r_squared"]

print(f"Critical Force (CF):  {cf:.1f} % MVC")
print(f"W' (anaerobic cap.):  {wp:.0f} %MVC\u00b7s")
print(f"R²:                   {r2:.4f}")

# Convert CF to absolute force
cf_kg = cf / 100 * MVC7_KG
print(f"\nCF in absolute terms: {cf_kg:.1f} kg")

## Step 2 — Visualize the Linear Regression

Plotting **cumulative work** (intensity × time) on the y-axis against
**time** on the x-axis yields a straight line whose:

- **Slope** = CF (% MVC)
- **y-intercept** = W′ (% MVC · s)

A high R² confirms the critical force model fits the data well.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Compute work for each trial
times = np.array(DURATIONS)
work  = np.array(INTENSITIES) * times  # %MVC · s

# Regression line
t_fit = np.linspace(0, max(times) * 1.1, 200)
w_fit = cf * t_fit + wp

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(times, work, s=100, zorder=5, color="#e74c3c", edgecolors="k", label="Test trials")
ax.plot(t_fit, w_fit, "--", color="#2c3e50", lw=2,
        label=f"Fit: Work = {cf:.1f}·t + {wp:.0f}  (R²={r2:.4f})")

# Annotate each point
for i, (t, w, pct) in enumerate(zip(times, work, INTENSITIES)):
    ax.annotate(f"{pct}% MVC\n{t}s", (t, w),
                textcoords="offset points", xytext=(10, 10), fontsize=9)

ax.set_xlabel("Cumulative hang time (s)", fontsize=12)
ax.set_ylabel("Cumulative work (%MVC · s)", fontsize=12)
ax.set_title("Critical Force — Work vs. Time", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Step 3 — CF / Bodyweight Ratio

The **CF/BW ratio** combines finger strength (MVC-7 as % BW) with aerobic
endurance (CF as % MVC) into a single number that correlates with climbing
grade (Giles et al. 2019).

$$\text{CF/BW} = \frac{CF_{\%MVC}}{100} \times \frac{MVC7_{\%BW}}{100}$$

In [ ]:
from climbing_science.endurance import cf_mvc_ratio

ratio = cf_mvc_ratio(cf, MVC7_PCT_BW)
print(f"CF/BW ratio: {ratio:.2f}")
print(f"  (CF = {cf:.1f}% of MVC-7 = {MVC7_PCT_BW:.0f}% BW)")

## Step 4 — Interpret Your CF/BW Ratio

How does your ratio compare to published benchmarks?

In [ ]:
from climbing_science.endurance import interpret_cf_ratio

interpretation = interpret_cf_ratio(ratio)
print(f"Level:       {interpretation['level']}")
print(f"Description: {interpretation['description']}")

## Step 5 — Endurance Classification

Beyond the CF/BW ratio, the combination of CF (%) and W′ tells you about
your *endurance profile*: are you a "diesel engine" (high CF, low W′) or
a "sprinter" (low CF, high W′)?

In [ ]:
from climbing_science.endurance import classify_endurance

classification = classify_endurance(cf, wp)
for key, value in classification.items():
    print(f"{key}: {value}")

## Step 6 — Predict Time-to-Failure

The critical force model predicts how long you can sustain any force above
CF before exhaustion:

$$t_{\text{lim}} = \frac{W'}{F_{\%MVC} - CF_{\%MVC}}$$

Below CF the model predicts infinite endurance (no W′ depletion). Above CF,
higher forces drain the battery faster.

In [ ]:
from climbing_science.endurance import time_to_failure

# Predict TtF for a range of intensities above CF
test_intensities = list(range(int(cf) + 5, 100, 5))

print("Predicted Time-to-Failure")
print("-" * 35)
print(f"{'Intensity (%MVC)':>18}  {'TtF (s)':>8}  {'TtF (min)':>10}")
print("-" * 35)
for intensity in test_intensities:
    ttf = time_to_failure(intensity, cf, wp)
    print(f"{intensity:>18}  {ttf:>8.0f}  {ttf/60:>10.1f}")

In [ ]:
# Plot the predicted TtF curve
intensities_curve = np.linspace(cf + 1, 100, 300)
ttf_curve = [time_to_failure(i, cf, wp) for i in intensities_curve]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(intensities_curve, np.array(ttf_curve) / 60, color="#2980b9", lw=2.5)
ax.axvline(cf, color="#e74c3c", ls="--", lw=1.5, label=f"CF = {cf:.1f}% MVC")

# Mark the original test points
for intensity, duration in zip(INTENSITIES, DURATIONS):
    ax.plot(intensity, duration / 60, "o", ms=10, color="#e74c3c",
            edgecolors="k", zorder=5)
    ax.annotate(f"{duration}s", (intensity, duration / 60),
                textcoords="offset points", xytext=(8, 5), fontsize=9)

ax.set_xlabel("Force (% MVC)", fontsize=12)
ax.set_ylabel("Time to failure (min)", fontsize=12)
ax.set_title("Predicted Time-to-Failure Curve", fontsize=14)
ax.set_xlim(cf - 5, 100)
ax.set_ylim(0, max(ttf_curve) / 60 * 1.1)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## What Trains Critical Force?

CF reflects the **aerobic capacity** of the forearm — the ability to
sustain force via oxidative metabolism while clearing metabolic byproducts.
Training methods that improve CF:

### 1. Endurance Repeaters
- **Protocol**: 7 s on / 3 s off at 30–45 % MVC for 10–20+ minutes
- **Goal**: Maximize time under tension at sub-CF intensities
- **Mechanism**: Capillary density ↑, mitochondrial density ↑, blood flow ↑

### 2. Density Hangs ("SubHangs")
- **Protocol**: Longer hangs (20–45 s) at 30–40 % MVC with short rest
- **Goal**: Sustained aerobic loading without occlusion
- **Mechanism**: Oxidative enzyme activity ↑, fatigue resistance ↑

### 3. ARC Training (on the wall)
- **Protocol**: 20–45 minutes of easy, continuous climbing well below flash level
- **Goal**: Aerobic base for the forearm
- **Mechanism**: Same as above, more sport-specific

> **Key principle**: CF training happens *below* CF. You are training the
> aerobic system, not depleting W′.

## W′ — Your Anaerobic Battery

W′ (pronounced "W-prime") represents the **total amount of work you can
perform above CF** before exhaustion. Think of it as a finite battery:

- **Draining**: Every second spent above CF drains W′ at a rate of
  *(current force − CF)* per second
- **Recharging**: During rest (or when force drops below CF), W′
  reconstitutes — but the recharge rate depends on how depleted it is
  (Jones et al. 2010)

### What affects W′?

| Factor | Effect on W′ |
|--------|-------------|
| Anaerobic glycolytic capacity | ↑ W′ |
| PCr stores in forearm | ↑ W′ |
| High-intensity repeater training | ↑ W′ |
| Endurance training alone | Minimal effect |

### Training W′

To increase W′, train **above CF** with sufficient rest for partial
reconstitution:
- **High-intensity repeaters**: 7 s on / 3 s off at 60–80 % MVC, 4–6 sets of 3–5 min
- **Max hangs**: 10 s at 80–90 % MVC, long rest (≥ 3 min)
- **Bouldering**: Short, intense problems with full recovery

*References: Poole et al. 2016; Jones et al. 2010; Giles et al. 2019.*

## References

- **Monod, H. & Scherrer, J.** (1965). The work capacity of a synergic
  muscular group. *Ergonomics*, 8(3), 329–338.
- **Giles, D., Hartley, C., Massey, H., Sherrer, S., Sherrington, A.,
  & Mayson, T.** (2019). An all-out test to determine finger flexor
  critical force in rock climbers. *International Journal of Sports
  Physiology and Performance*, 14(10), 1378–1388.
- **Poole, D. C., Burnley, M., Vanhatalo, A., Rossiter, H. B., &
  Jones, A. M.** (2016). Critical power: An important fatigue threshold
  in exercise physiology. *Medicine and Science in Sports and Exercise*,
  48(11), 2320–2334.
- **Jones, A. M., Vanhatalo, A., Burnley, M., Morton, R. H., &
  Poole, D. C.** (2010). Critical power: Implications for determination
  of VO2max and exercise tolerance. *Medicine and Science in Sports and
  Exercise*, 42(10), 1876–1890.